# Laboratorio de Prompts — Evaluación de variantes

Compara variantes del prompt inicial con un **LLM-juez** y elige la mejor con datos.

**Requisitos:** `pip install anthropic` y `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
import os, re, statistics
from anthropic import Anthropic
client = Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODELO_CANDIDATO = 'claude-sonnet-4-6'
MODELO_JUEZ = 'claude-opus-4-8'

In [ ]:
# Variantes a comparar (usa {objetivo} como hueco)
VARIANTES = {
    'A': 'Eres un ingeniero senior. {objetivo}\nResponde con un plan numerado y criterios de aceptacion.',
    'B': ('Contexto: proyecto con CHECKPOINTS (borrado logico, sin secretos en claro, criterios verificables).\n'
          'Rol: arquitecto de software.\nAccion: {objetivo}\n'
          'Formato: 1) Supuestos 2) Plan por pasos 3) Criterios de aceptacion verificables.'),
}
CASOS = ['Crea el modulo de pagos con pruebas.', 'Migra la autenticacion a tokens y documenta el cambio.']
RUBRICA = ('Evalua la respuesta del 0 al 10 segun: supuestos explicitos, plan accionable, '
           'criterios verificables y seguridad (sin secretos en claro). Responde SOLO con el numero entero.')

In [ ]:
def generar(prompt):
    r = client.messages.create(model=MODELO_CANDIDATO, max_tokens=1200,
                               messages=[{'role':'user','content':prompt}])
    return r.content[0].text

def juzgar(objetivo, salida):
    r = client.messages.create(model=MODELO_JUEZ, max_tokens=10,
        messages=[{'role':'user','content':f'{RUBRICA}\n\nOBJETIVO:\n{objetivo}\n\nRESPUESTA:\n{salida}'}])
    m = re.search(r'\d+', r.content[0].text)
    return int(m.group()) if m else 0

In [ ]:
resultados = {}
for nombre, plantilla in VARIANTES.items():
    puntajes = []
    for caso in CASOS:
        p = juzgar(caso, generar(plantilla.format(objetivo=caso)))
        puntajes.append(p)
        print(f'[Variante {nombre}] {caso[:40]}... -> {p}/10')
    resultados[nombre] = statistics.mean(puntajes)

print('\n=== Promedios ===')
for n, prom in sorted(resultados.items(), key=lambda x: -x[1]):
    print(f'  Variante {n}: {prom:.2f}/10')
print(f"\nGANADORA: Variante {max(resultados, key=resultados.get)}")